# Experiment Analysis — 01 Reward Ablation
This notebook executes the full reward-ablation family via `src.workflows.experiment_workflow`, validates all variant artifacts, and summarizes risk/return outcomes.

Design goal: isolate how incremental reward components affect learned behavior and out-of-sample metrics.

In [1]:
from __future__ import annotations

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from src.utils.config_loader import resolve_config
from src.utils.seed import set_global_seed
from src.utils.artifact_manager import ArtifactManager
from src.workflows.data_workflow import run_data_workflow
from src.workflows.experiment_workflow import run_experiment_workflow

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
BASE_CFG = resolve_config(root=str(ROOT))
SEED = int(BASE_CFG.get('training', {}).get('random_seed', 42))
set_global_seed(SEED, deterministic_torch=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PAIR = 'EURUSD'
OUTPUTS_ROOT = str(ROOT / 'outputs')
EXPERIMENT_NAME = '01_reward_ablation'
RUN_EXPERIMENT = True
am = ArtifactManager(OUTPUTS_ROOT)

2026-03-13 03:38:39 | src.utils.seed | INFO | Global seed set to 42 (deterministic_torch=True)


In [ ]:
data = run_data_workflow(BASE_CFG, pairs=[PAIR], root=str(ROOT))
train_df = data[PAIR]['train']
test_df = data[PAIR]['test']

if RUN_EXPERIMENT:
    exp_results = run_experiment_workflow(
        root=ROOT,
        experiment_name=EXPERIMENT_NAME,
        train_df=train_df,
        test_df=test_df,
        outputs_root=OUTPUTS_ROOT,
    )
else:
    exp_results = {}

print(f'Experiment: {EXPERIMENT_NAME}')
print(f'Variants executed: {len(exp_results)}')
print('Variant names:', sorted(exp_results.keys()))

## Artifact integrity checks
Each variant must produce canonical artifacts: resolved config, checkpoint, final model, and train/test metric files.

In [ ]:
exp_root = Path(OUTPUTS_ROOT) / 'experiments' / EXPERIMENT_NAME
variant_dirs = sorted([d for d in exp_root.iterdir() if d.is_dir() and d.name != 'summary']) if exp_root.exists() else []

required_paths = [
    Path('resolved_config.yaml'),
    Path('checkpoints/checkpoint_latest.pt'),
    Path('checkpoints/training_state.json'),
    Path('models/model_final.pt'),
    Path('metrics/train/reward_curve.csv'),
    Path('metrics/train/loss_curve.csv'),
]

checks = []
for vd in variant_dirs:
    missing = [str(p) for p in required_paths if not (vd / p).exists()]
    checks.append({'variant': vd.name, 'missing_count': len(missing), 'missing': '; '.join(missing)})
    assert len(missing) == 0, f'Missing artifacts for {vd.name}: {missing}'

artifact_check_df = pd.DataFrame(checks)
display(artifact_check_df if not artifact_check_df.empty else pd.DataFrame([{'status': 'no variants found'}]))

## Aggregate variant metrics and visualize outcomes

In [ ]:
def read_performance_summary(path: Path) -> dict:
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    if {'metric', 'value'}.issubset(df.columns):
        return {str(r.metric): float(r.value) for r in df.itertuples(index=False)}
    return {}

rows = []
for vd in variant_dirs:
    perf = read_performance_summary(vd / 'tables' / 'test' / 'performance_summary.csv')
    if not perf:
        perf = read_performance_summary(vd / 'tables' / 'train' / 'performance_summary.csv')
    rows.append({'variant': vd.name, **perf})

variant_metrics_df = pd.DataFrame(rows).sort_values('variant') if rows else pd.DataFrame()
display(variant_metrics_df.head(20))

if not variant_metrics_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    if 'cumulative_return' in variant_metrics_df.columns:
        axes[0].bar(variant_metrics_df['variant'], variant_metrics_df['cumulative_return'])
        axes[0].set_title('Cumulative Return by Variant')
        axes[0].tick_params(axis='x', rotation=45)
    if 'sharpe_ratio' in variant_metrics_df.columns:
        axes[1].bar(variant_metrics_df['variant'], variant_metrics_df['sharpe_ratio'])
        axes[1].set_title('Sharpe Ratio by Variant')
        axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

summary_dir = exp_root / 'summary'
(summary_dir / 'tables').mkdir(parents=True, exist_ok=True)
(summary_dir / 'figures').mkdir(parents=True, exist_ok=True)
if not variant_metrics_df.empty:
    variant_metrics_df.to_csv(summary_dir / 'tables' / 'variant_metrics_notebook.csv', index=False)

## Reproducibility note
Set `RUN_DETERMINISM_CHECK = True` below to rerun the same experiment family and compare variant-level summary stability under fixed seeds.

In [ ]:
RUN_DETERMINISM_CHECK = False

if RUN_DETERMINISM_CHECK:
    rerun = run_experiment_workflow(
        root=ROOT,
        experiment_name=EXPERIMENT_NAME,
        train_df=train_df,
        test_df=test_df,
        outputs_root=OUTPUTS_ROOT,
    )
    before = sorted(exp_results.keys())
    after = sorted(rerun.keys())
    assert before == after, 'Variant set changed across deterministic rerun'
    print('Determinism check: variant set stable ✅')
else:
    print('Determinism check skipped (set RUN_DETERMINISM_CHECK=True to enable).')

print(f'✅ {EXPERIMENT_NAME} notebook complete.')